In [51]:
#import packages
import numpy as np
import scipy
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
import os
import yaml
import itertools
import shutil
from datetime import datetime
from pathlib import Path
from openpyxl.styles import PatternFill
from openpyxl import load_workbook
from collections import defaultdict
import sys
from openpyxl.styles import Border, Side
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter
from numpy._core.numeric import indices
from tqdm import tqdm
from functools import lru_cache
from collections import deque
import networkx as nx
from collections import Counter

#set working dir
os.chdir("/Users/quinnmackay/Documents/GitHub/BICC/Antarctic Chronology Accuracy Project")

In [52]:
#combination analysis first:
# load cores
project = 'all_tiepoints' #DO NOT CHANGE IN THIS FILE!
output_dir = 'table_out/'

# get all link combos
with open(f'/Users/quinnmackay/Documents/GitHub/BICC/Antarctic Chronology Accuracy Project/{project}/parameters.yml') as f:
    data = yaml.safe_load(f)
list_sites = data["list_sites"]
pairs = [f"{a}-{b}" for a, b in itertools.combinations(list_sites, 2)]

#define error margins
error_margin = 0.15

big_table = pd.DataFrame()
all_links_count = {}
all_links_foragesort = {}
all_links_total = {}

for core in list_sites: # loop through each core
    for comparison_core in list_sites: # loop through each core other than the initial load
        pair = f"{core}-{comparison_core}"
        if core != comparison_core and pair in pairs: # make sure not the same core and we skip non-existent linkages
            pair_dir = Path(f'/Users/quinnmackay/Documents/GitHub/BICC/Antarctic Chronology Accuracy Project/{project}/{pair}')

            # Check: directory exists AND contains at least one .txt file
            txt_files = list(pair_dir.glob("*.txt"))
            if not pair_dir.is_dir() or not txt_files:
                continue

            dfs=[] #load all text files into one
            for txt in txt_files:
                df = pd.read_csv(txt, sep="\t", comment="#")
                dfs.append(df)
    
            num_files = len(dfs)
            load_data = pd.concat(dfs, ignore_index=True)
            original_rows = len(load_data)

            drop_rows = []
            drop_rows_merge = set()
            new_merged_rows = []
            for idx, row in load_data.iterrows():

                mask1 = abs(row['depth1'] - load_data['depth1']) <= error_margin
                mask1[idx] = False
                mask2 = abs(row['depth2'] - load_data['depth2']) <= error_margin 
                mask2[idx] = False

                close_points = load_data[mask1 & mask2]
                num_close = len(close_points)
                close_idxs = load_data.index[mask1 & mask2]

                if num_close > 0:
                    refs = [load_data.at[idx, 'reference']] + [load_data.at[i, 'reference'] for i in close_idxs] #adjoin references
                    merged_ref = "; ".join(str(r) for r in refs if pd.notna(r))

                    depth1_vals = [load_data.at[idx, 'depth1']] + [load_data.at[i, 'depth1'] for i in close_idxs]
                    merged_depth1 = np.round(np.mean(depth1_vals), 4)

                    depth2_vals = [load_data.at[idx, 'depth2']] + [load_data.at[i, 'depth2'] for i in close_idxs]
                    merged_depth2 = np.round(np.mean(depth2_vals), 4)

                    new_merged_rows.append({'reference': merged_ref, 'depth1': merged_depth1, 'depth2': merged_depth2}) #create new merged row

                    drop_rows_merge.add(idx)
                    for i in close_idxs:
                        drop_rows.append(i)
                        if drop_rows.count(i) >= num_files:
                            print(f'WARNING: Row {load_data.at[i, 'depth1']} | {load_data.at[i, 'depth2']} for {pair}. Reference {load_data.at[i, 'reference']}.')
                            print(f'Called by row {load_data.at[idx, 'depth1']} | {load_data.at[idx, 'depth2']} from reference {load_data.at[idx, 'reference']}.')

            # drop duplicate rows
            drop_rows = set(drop_rows).union(drop_rows_merge)
            load_data = load_data.drop(index=drop_rows).reset_index(drop=True)
            # add merged rows
            merged_df = pd.DataFrame(new_merged_rows)
            load_data = pd.concat([load_data, merged_df], ignore_index=True)
            load_data.drop_duplicates(subset=['depth1', 'depth2'], inplace=True)
            load_data = load_data.reset_index(drop=True)

            load_data = load_data.sort_values(by=['depth1']).reset_index(drop=True)
        
            #set up pair code stuff
            load_data[f"{pair}_code"] = [f"{pair}_{idx}" for idx in load_data.index]

            #save all the links for this pair
            all_links_total[f'{pair}'] = load_data[['depth1', 'depth2']].copy(deep=True)
            all_links_total[f'{pair}'] = all_links_total[f'{pair}'].rename(columns={
                'depth1': pair.split("-")[0],
                'depth2': pair.split("-")[1]})

            # rename to create unique columns for this pair
            load_data = load_data.rename(columns={
                'depth1': f"{pair}_{core}",
                'depth2': f"{pair}_{comparison_core}",
                'reference': f"{pair}_reference",
            })

            print(f"Processed pair {pair}, total points after merging: {len(load_data)}, ({original_rows} original total rows)")
            # append rows (block)
            big_table = pd.concat([big_table, load_data], axis=0, ignore_index=True)

#if core doesn't exist in all_links_count, add it with 0 val
for core in list_sites:
    if core not in all_links_count:
        all_links_count[core] = 0

big_table.to_csv(f'/Users/quinnmackay/Desktop/output2.csv', index=False)

Called by row 1663.938 | 1609.565 from reference Seierstad2014_Rasmussen2014.
Processed pair GISP2-GRIP, total points after merging: 823, (918 original total rows)
Processed pair GISP2-NEEM, total points after merging: 194, (194 original total rows)
Processed pair GISP2-NG1, total points after merging: 290, (290 original total rows)
Processed pair GISP2-NG2, total points after merging: 614, (725 original total rows)
Processed pair GISP2-DF, total points after merging: 118, (118 original total rows)
Processed pair GISP2-TALDICE, total points after merging: 89, (89 original total rows)
Processed pair GISP2-EDC, total points after merging: 138, (138 original total rows)
Processed pair GISP2-EDML, total points after merging: 155, (155 original total rows)
Processed pair GISP2-WDC, total points after merging: 286, (299 original total rows)
Processed pair GRIP-NEEM, total points after merging: 396, (396 original total rows)
Processed pair GRIP-NG1, total points after merging: 519, (587 origi

In [53]:
big_table

,GISP2-GRIP_GISP2,GISP2-GRIP_GRIP,GISP2-GRIP_reference,GISP2-GRIP_code,GISP2-NEEM_GISP2,GISP2-NEEM_NEEM,GISP2-NEEM_reference,GISP2-NEEM_code,GISP2-NG1_GISP2,GISP2-NG1_NG1,...,EDC-EDML_reference,EDC-EDML_code,EDC-WDC_EDC,EDC-WDC_WDC,EDC-WDC_reference,EDC-WDC_code,EDML-WDC_EDML,EDML-WDC_WDC,EDML-WDC_reference,EDML-WDC_code
0,33.047,30.620,Seierstad2014_Rasmussen2014,GISP2-GRIP_0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,46.330,43.153,Seierstad2014_Rasmussen2014,GISP2-GRIP_1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,54.507,50.887,Seierstad2014_Rasmussen2014,GISP2-GRIP_2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,57.160,53.407,Seierstad2014_Rasmussen2014,GISP2-GRIP_3,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,58.228,54.510,Seierstad2014_Rasmussen2014,GISP2-GRIP_4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14122,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1756.750,3392.42,Svensson_Links,EDML-WDC_969
14123,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1762.070,3394.59,Svensson_Links,EDML-WDC_970
14124,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1766.861,3397.10,Svensson_Links,EDML-WDC_971
14125,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1771.930,3399.99,Svensson_Links,EDML-WDC_972


In [54]:
# core1 = 'WDC'
# core2 = 'DF'

# match = next((p for p in pairs if set(p.split('-')) == {core1, core2})) #get the correct order of match
for match in pairs:
    match1 = match.split('-')[0]
    match2 = match.split('-')[1]

    try:
        core1_data = big_table[f'{match}_{match1}'].dropna().reset_index(drop=True)
    except KeyError:
        print(f"Warning: {match}_{match1} not found in big_table. Skipping.")
        continue
    core2_data = big_table[f'{match}_{match2}'].dropna().reset_index(drop=True)

    output_dir = Path('/Users/quinnmackay/Documents/GitHub/BICC/Antarctic Chronology Accuracy Project/Plot_Sulfate/tiepoints') / match
    output_dir.mkdir(parents=True, exist_ok=True)

    core1_data.to_csv(f'{output_dir}/{match1}.txt', index=False)
    core2_data.to_csv(f'{output_dir}/{match2}.txt', index=False)